# Chronos 2 Foundation Model Forecasting

This notebook implements zero-shot univariate forecasting for hourly bike rental demand using Chronos 2.

The final setup uses only the historical target variable `cnt`. Because we do not tune model variants in this version, the model is fitted on the full available development data (`train + validation`) and evaluated once on the test set.

Databricks execution note: run this notebook on a GPU-enabled cluster with AutoGluon TimeSeries installed. If AutoGluon or Chronos is missing, install it first with `%pip install -U "autogluon.timeseries[chronos]" "chronos-forecasting>=2.0"`, restart Python, and then rerun the notebook.


In [0]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

In [0]:
def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path] + list(current_path.parents)

    for candidate_path in candidate_paths:
        if (candidate_path / "data" / "processed" / "train.csv").exists():
            return candidate_path

    raise FileNotFoundError(
        "Could not find project root. Run this notebook from the repository "
        "or make sure data/processed/train.csv exists."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FORECAST_DIR = PROJECT_ROOT / "outputs" / "forecasts"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

os.makedirs(FORECAST_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

TARGET = "cnt"
DATE_COLUMN = "dteday"
HOUR_COLUMN = "hr"
TIMESTAMP_COLUMN = "timestamp"

ID_COLUMN = "item_id"
ITEM_ID = "bike_rentals"

FREQUENCY = "h"
FORECAST_HORIZON = 24
EVALUATION_METRIC = "MASE"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# Load Chronological Data Splits

The train, validation and test sets were created previously during the preprocessing stage. In this simplified Chronos setup, train and validation are later combined as the full historical context, while the test set stays untouched until the final evaluation.


In [0]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
val = pd.read_csv(f"{DATA_DIR}/val.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")


def add_hourly_timestamp(df):
    df = df.copy()

    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
    df[TIMESTAMP_COLUMN] = (
        df[DATE_COLUMN] + pd.to_timedelta(df[HOUR_COLUMN].astype(int), unit="h")
    )

    df = df.sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)

    if df[TIMESTAMP_COLUMN].duplicated().any():
        duplicated_timestamps = df.loc[
            df[TIMESTAMP_COLUMN].duplicated(), TIMESTAMP_COLUMN
        ].head()

        raise ValueError(
            f"Duplicate hourly timestamps found: {duplicated_timestamps.tolist()}"
        )

    return df


train = add_hourly_timestamp(train)
val = add_hourly_timestamp(val)
test = add_hourly_timestamp(test)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

print("Duplicate timestamps:")
print("train:", train[TIMESTAMP_COLUMN].duplicated().sum())
print("val:", val[TIMESTAMP_COLUMN].duplicated().sum())
print("test:", test[TIMESTAMP_COLUMN].duplicated().sum())

In [0]:
required_columns = [TARGET]

for split_name, split_df in {
    "train": train,
    "validation": val,
    "test": test
}.items():
    missing_columns = set(required_columns) - set(split_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in {split_name} split: {missing_columns}"
        )

In [0]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(val), len(test)],
    "start": [
        train[TIMESTAMP_COLUMN].min(),
        val[TIMESTAMP_COLUMN].min(),
        test[TIMESTAMP_COLUMN].min()
    ],
    "end": [
        train[TIMESTAMP_COLUMN].max(),
        val[TIMESTAMP_COLUMN].max(),
        test[TIMESTAMP_COLUMN].max()
    ],
    "target_mean": [
        train[TARGET].mean(),
        val[TARGET].mean(),
        test[TARGET].mean()
    ]
})

split_summary


In [0]:
train.head()

# Prepare Data For AutoGluon Chronos 2

Chronos 2 will be used through AutoGluon TimeSeries. In this step, we prepare one clean univariate modelling dataset with the target variable `cnt` and the hourly timestamp required by AutoGluon.


In [0]:
try:
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
except ImportError as error:
    raise ImportError(
        "AutoGluon TimeSeries and Chronos are required to run this notebook. "
        "Install them before running Chronos 2, for example with: "
        "%pip install -U \"autogluon.timeseries[chronos]\" \"chronos-forecasting>=2.0\""
    ) from error


# Select Modelling Columns

For this version we keep only the univariate Chronos 2 setup. The model receives the historical target variable `cnt`; engineered lag and rolling features are not used because Chronos builds its own representation from the sequence history.


In [0]:
UNIVARIATE_COLUMNS = [TARGET]

EXCLUDED_ENGINEERED_FEATURES = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_48",
    "lag_168",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168"
]


In [0]:
selected_columns = pd.DataFrame({
    "column_group": ["target"],
    "column_name": UNIVARIATE_COLUMNS
})

selected_columns


In [0]:
required_columns = set(UNIVARIATE_COLUMNS + [TIMESTAMP_COLUMN])

for split_name, split_df in {
    "train": train,
    "validation": val,
    "test": test
}.items():
    missing_columns = required_columns - set(split_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in {split_name} split: {missing_columns}"
        )


# Convert Data To AutoGluon Format

AutoGluon TimeSeries needs a long dataframe with an item identifier and timestamp. Because we have one time series, the `item_id` is constant for all observations. We prepare the final training data from `train + validation`, because no validation-based model selection is used in this simplified Chronos setup.


In [0]:
def prepare_autogluon_dataframe(df, modelling_columns):
    model_df = df[[TIMESTAMP_COLUMN] + modelling_columns].copy()
    model_df[ID_COLUMN] = ITEM_ID

    return model_df[[ID_COLUMN, TIMESTAMP_COLUMN] + modelling_columns]


def to_timeseries_dataframe(df):
    return TimeSeriesDataFrame.from_data_frame(
        df,
        id_column=ID_COLUMN,
        timestamp_column=TIMESTAMP_COLUMN
    )

In [0]:
training_context = pd.concat([
    train,
    val
], ignore_index=True)

training_context = (
    training_context
    .sort_values(TIMESTAMP_COLUMN)
    .reset_index(drop=True)
)

train_univariate_df = prepare_autogluon_dataframe(
    training_context,
    UNIVARIATE_COLUMNS
)

train_univariate_data = to_timeseries_dataframe(
    train_univariate_df
)


In [0]:
print("Final training context shape:", training_context.shape)
print("Univariate training data shape:", train_univariate_data.shape)

train_univariate_df.head()


# Evaluation Metrics

We use MASE as the main validation metric for choosing the better Chronos 2 setup. MAE and RMSE are also calculated, because they make the results easier to compare with the other models in the project. MASE is calculated with a seasonal naive benchmark using a 24-hour seasonal period.


In [0]:
POINT_FORECAST_COLUMN = "mean"


def rmse(
    y_true,
    y_pred
):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


def mase(
    y_true,
    y_pred,
    train_series,
    seasonal_period=24
):
    seasonal_naive_error = np.mean(
        np.abs(
            train_series[seasonal_period:].values
            - train_series[:-seasonal_period].values
        )
    )

    if seasonal_naive_error == 0:
        raise ValueError(
            "MASE cannot be calculated because the seasonal naive forecast error is zero."
        )

    model_mae = mean_absolute_error(
        y_true,
        y_pred
    )

    return model_mae / seasonal_naive_error


def evaluate_forecasts(
    y_true,
    y_pred,
    train_series
):
    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "RMSE": rmse(
            y_true,
            y_pred
        ),
        "MASE": mase(
            y_true,
            y_pred,
            train_series,
            seasonal_period=FORECAST_HORIZON
        )
    }


# Expanding-Window Test Logic

The final evaluation is done on the test set in 24-hour expanding-window blocks. In every window, Chronos receives all history available up to that point and forecasts the next 24 hours.


In [0]:
def get_forecast_starts(
    evaluation_df,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON
):
    return range(
        0,
        len(evaluation_df) - forecast_horizon,
        step_size
    )


def summarize_expanding_window_setup(
    evaluation_df,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON
):
    forecast_starts = list(
        get_forecast_starts(
            evaluation_df,
            forecast_horizon,
            step_size
        )
    )

    evaluated_rows = len(forecast_starts) * forecast_horizon
    non_evaluated_rows = len(evaluation_df) - evaluated_rows

    return pd.DataFrame({
        "forecast_horizon": [forecast_horizon],
        "step_size": [step_size],
        "forecast_windows": [len(forecast_starts)],
        "available_rows": [len(evaluation_df)],
        "evaluated_rows": [evaluated_rows],
        "non_evaluated_rows": [non_evaluated_rows]
    })

In [0]:
test_window_summary = summarize_expanding_window_setup(
    test,
    FORECAST_HORIZON
)

test_window_summary.assign(split="test")[[
    "split",
    "forecast_horizon",
    "step_size",
    "forecast_windows",
    "available_rows",
    "evaluated_rows",
    "non_evaluated_rows"
]]


In [0]:
def extract_point_forecast(
    predictions,
    point_forecast_column=POINT_FORECAST_COLUMN
):
    if point_forecast_column not in predictions.columns:
        raise ValueError(
            f"Column '{point_forecast_column}' is not available in predictions. "
            f"Available columns: {list(predictions.columns)}"
        )

    point_forecast = predictions[point_forecast_column]

    if isinstance(point_forecast.index, pd.MultiIndex):
        try:
            point_forecast = point_forecast.xs(
                ITEM_ID,
                level=ID_COLUMN
            )
        except KeyError:
            point_forecast = point_forecast.xs(
                ITEM_ID,
                level=0
            )

    return point_forecast

In [0]:
def run_expanding_window_forecast(
    predictor,
    context_df,
    evaluation_df,
    modelling_columns,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON,
    point_forecast_column=POINT_FORECAST_COLUMN
):
    all_results = []

    forecast_starts = get_forecast_starts(
        evaluation_df,
        forecast_horizon,
        step_size
    )

    for start in forecast_starts:
        history = pd.concat([
            context_df,
            evaluation_df.iloc[:start]
        ]).copy()

        history_data = to_timeseries_dataframe(
            prepare_autogluon_dataframe(
                history,
                modelling_columns
            )
        )

        predictions = predictor.predict(
            history_data,
            random_seed=RANDOM_SEED
        )

        point_predictions = extract_point_forecast(
            predictions,
            point_forecast_column
        )

        actual_future = evaluation_df.iloc[
            start:start + forecast_horizon
        ][[TIMESTAMP_COLUMN, TARGET]].copy()

        n_forecasts = min(
            len(actual_future),
            len(point_predictions)
        )

        actual_future = actual_future.iloc[:n_forecasts].copy()
        aligned_predictions = point_predictions.iloc[:n_forecasts]

        window_results = pd.DataFrame({
            TIMESTAMP_COLUMN: actual_future[TIMESTAMP_COLUMN].values,
            "window_start": actual_future[TIMESTAMP_COLUMN].iloc[0],
            "actual": actual_future[TARGET].values,
            "prediction": aligned_predictions.values
        })

        all_results.append(window_results)

    if not all_results:
        return pd.DataFrame(
            columns=[TIMESTAMP_COLUMN, "window_start", "actual", "prediction"]
        )

    return pd.concat(
        all_results,
        ignore_index=True
    )


In [0]:
def evaluate_expanding_window_results(
    forecast_df,
    train_series
):
    return evaluate_forecasts(
        forecast_df["actual"],
        forecast_df["prediction"],
        train_series
    )

# Zero-Shot Univariate Chronos 2

The final setup uses only the historical values of the target variable `cnt`. We use the standard Chronos 2 preset in zero-shot mode, so the model is not fine-tuned on our dataset. The combined `train + validation` data is passed to `.fit()` because AutoGluon needs it to infer time-series metadata and save the predictor state, not because Chronos 2 learns new weights here.


In [0]:
CHRONOS_PRESET = "chronos2"

MODEL_DIR = PROJECT_ROOT / "outputs" / "models"

os.makedirs(MODEL_DIR, exist_ok=True)

In [0]:
univariate_predictor = TimeSeriesPredictor(
    prediction_length=FORECAST_HORIZON,
    target=TARGET,
    eval_metric=EVALUATION_METRIC,
    freq=FREQUENCY,
    path=str(MODEL_DIR / "chronos2_univariate_final"),
    verbosity=2
).fit(
    train_univariate_data,
    presets=CHRONOS_PRESET,
    enable_ensemble=False,
    random_seed=RANDOM_SEED
)


# Final Test Evaluation

The univariate Chronos 2 setup is evaluated on the test set only once. The available history before the test set is `train + validation`, and the test set is forecasted in 24-hour expanding-window blocks.


In [0]:
chronos2_test_forecasts = run_expanding_window_forecast(
    predictor=univariate_predictor,
    context_df=training_context,
    evaluation_df=test,
    modelling_columns=UNIVARIATE_COLUMNS,
    step_size=FORECAST_HORIZON
)

chronos2_test_metrics = evaluate_expanding_window_results(
    chronos2_test_forecasts,
    training_context[TARGET]
)

chronos2_test_metrics


In [0]:
plt.figure(figsize=(14, 6))

plt.plot(
    chronos2_test_forecasts["actual"].values[:168],
    label="Actual"
)

plt.plot(
    chronos2_test_forecasts["prediction"].values[:168],
    label="Chronos 2 Forecast"
)

plt.title("Chronos 2 Test Forecasts (First 168 Hours)")
plt.xlabel("Forecast Horizon")
plt.ylabel("Bike Rentals")
plt.legend()

plt.show()

# Save Metrics And Results

The final test metrics and final test forecasts are saved to the project output folders. The forecast file keeps the same simple structure as the other model outputs: actual values and predictions.


In [0]:
chronos2_test_metrics_df = pd.DataFrame({
    "metric": list(chronos2_test_metrics.keys()),
    "value": list(chronos2_test_metrics.values())
})

chronos2_test_metrics_df.to_csv(
    METRICS_DIR / "chronos2_test_metrics.csv",
    index=False
)

chronos2_test_forecasts[["actual", "prediction"]].to_csv(
    FORECAST_DIR / "chronos2_test_forecasts.csv",
    index=False
)

chronos2_test_metrics_df
